# 第3回A 演習: 言語モデルの仕組みと実装

スライド: [第3回A](../03a-llm/)

Colab で `ランタイム → ランタイムのタイプを変更 → T4 GPU` にしておく。
CPU でも動くが、生成に時間がかかる。

モデルの取得に数分かかる。先に先頭のセルを実行し、待っている間に課題を読む。

## 進め方

- `TODO` を埋めて、直下の確認セルを実行する
- 確認セル (`# [CHECK]`) は自分用。消しても教員の採点は別途走るが、残すこと

| 課題 | 配点 |
|---|---|
| 課題1 次のトークンの上位候補 | 20 |
| 課題2 greedy 生成 | 20 |
| 課題2-b 繰り返し率 | 10 |
| 課題3 温度付き top-k | 20 |
| 課題4 chat | 10 |
| 課題4-b 会話を続ける | 10 |
| 発展1 top-p | 5 |
| 発展2 繰り返しペナルティ | 5 |


In [ ]:
!pip install -q transformers torch


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL)
model.eval()

if torch.cuda.is_available():
    device = 'cuda'                        # Colab の GPU
elif torch.backends.mps.is_available():
    device = 'mps'                         # Apple Silicon の Mac
else:
    device = 'cpu'
model.to(device)
print(device, '/ 語彙数:', len(tok))


In [ ]:
# 以降で使うヘルパ
def encode(text):
    """文字列 -> トークン ID のテンソル"""
    return tok(text, return_tensors='pt').input_ids.to(device)


def next_logits(ids):
    """現在の系列から「次のトークンの logits」を取り出す"""
    with torch.no_grad():
        return model(ids).logits[0, -1]


print(encode('こんにちは').shape, next_logits(encode('こんにちは')).shape)


In [ ]:
#@title 自動セットアップ (実行するだけ。中身は隠しています)
# [SKIP] 採点の対象外
# ---- 自己確認 (check / score) の準備。1回実行すればよい ----
import pathlib
import urllib.request

if not pathlib.Path('autograde.py').exists():
    try:
        urllib.request.urlretrieve(
            'https://3tty0n.github.io/2026-scripting-lab/grading/autograde.py', 'autograde.py')
    except Exception as e:
        print('取得できませんでした:', e)

from autograde import check, score
TASK = '03a'


---
## 課題1: 次のトークンの候補を見る (20点)

文字列を受け取り、次に来るトークンの**上位 n 候補**を
`[(トークン文字列, 確率), ...]` の形で返す `top_candidates(text, n=10)` を書け。

- 確率の**降順**に並べる
- 確率は `float` にすること

**ヒント**: `next_logits` → `torch.softmax(..., dim=-1)` → `torch.topk(probs, n)`。
`topk` の返り値は `.values` と `.indices` を持つ。
ID を文字列に戻すのは `tok.decode([i])`。

### 例題

次のセルを実行して、動きを見る。

部品を1つずつ確かめる。下をそのまま実行してから、関数にまとめればよい。


In [ ]:
# 例題
lg = next_logits(encode('日本の首都は'))
print(lg.shape)                        # torch.Size([151936])  語彙ぶんの点数

probs = torch.softmax(lg, dim=-1)      # 点数 -> 確率 (合計 1)
top = torch.topk(probs, 3)             # 上位3件
print(top.values)                      # tensor([0.42, 0.11, 0.05])
print(top.indices)                     # tensor([100562, 3583, ...])

for p, i in zip(top.values.tolist(), top.indices.tolist()):
    print(repr(tok.decode([i])), p)    # '東京' 0.42 ...


`.tolist()` でテンソルから普通の Python の値に戻る。
`topk` は大きい順に返すので、並べ替えは要らない。


In [ ]:
def top_candidates(text, n=10):
    # TODO
    pass


for t, p in top_candidates('むかしむかし、あるところに'):
    print(f'{t!r:20} {p:.4f}')


In [ ]:
# [CHECK] q1
need('top_candidates', arity=2)

cand = top_candidates('日本の首都は', n=10)
assert isinstance(cand, list) and len(cand) == 10, len(cand)
assert all(isinstance(t, str) for t, _ in cand), 'トークンは文字列で返すこと'
probs = [float(p) for _, p in cand]
assert probs == sorted(probs, reverse=True), '確率の降順になっていない'
assert 0 < probs[0] <= 1.0, probs[0]
assert sum(probs) <= 1.0001, '確率の合計が1を超えている (softmax を通したか)'
assert len(top_candidates('こんにちは', n=3)) == 3

# 上位1件は logits の最大値と一致するはず
import torch as _t
lg = next_logits(encode('日本の首都は'))
assert cand[0][0] == tok.decode([int(_t.argmax(lg))]), '上位1件が argmax と違う'
print('OK')


---
## 課題2: greedy 生成 (20点)

毎回もっとも確率の高いトークンを選び続ける `generate_greedy(prompt, max_new_tokens)` を書け。

- 終端トークン (`tok.eos_token_id`) が出たら止める
- 返すのは**プロンプトを含めた全体**の文字列 (`skip_special_tokens=True`)

**ヒント**: `torch.argmax(logits)` / `torch.cat([ids, next_id.view(1, 1)], dim=1)`。

### 例題

次のセルを実行して、動きを見る。

1歩だけ進めてみる。ループはこれの繰り返し。


In [ ]:
# 例題
ids = encode('日本で一番高い山は')
print(ids.shape)                       # torch.Size([1, 8])  (1文, 8トークン)

lg = next_logits(ids)
next_id = torch.argmax(lg)             # 一番点数の高いトークン
print(next_id.item(), repr(tok.decode([next_id.item()])))

ids = torch.cat([ids, next_id.view(1, 1)], dim=1)   # 末尾にくっつける
print(ids.shape)                       # torch.Size([1, 9])
print(tok.decode(ids[0], skip_special_tokens=True))


`view(1, 1)` は形を `[1, 1]` に直すため。`ids` が `[1, n]` なので形をそろえる。
終わりの判定は `if next_id.item() == tok.eos_token_id: break`。
これを `ids` にくっつける**前**に置くこと。


In [ ]:
def generate_greedy(prompt, max_new_tokens=40):
    ids = encode(prompt)
    for _ in range(max_new_tokens):
        logits = next_logits(ids)
        # TODO
        pass
    return tok.decode(ids[0], skip_special_tokens=True)


print(generate_greedy('むかしむかし、あるところに'))


In [ ]:
# [CHECK] q2
need('generate_greedy', arity=2)

a = generate_greedy('日本で一番高い山は', 20)
b = generate_greedy('日本で一番高い山は', 20)
assert a == b, 'greedy なのに結果が変わっている'
assert a.startswith('日本で一番高い山は'), a[:30]
assert len(a) > len('日本で一番高い山は'), '何も生成されていない'

# max_new_tokens を守ること
short = generate_greedy('むかしむかし', 3)
n_new = len(tok(short).input_ids) - len(tok('むかしむかし').input_ids)
assert n_new <= 3, f'{n_new} トークン生成された'
assert generate_greedy('あ', 0) == 'あ'
print('OK')


### 課題2-b: 繰り返し率を測る (10点)

greedy は同じ言い回しのループに落ちやすい。それを**数値で**測る関数を書け。

`repetition_rate(text, n=3)` — テキストを空白で区切った単語列について、
**長さ `n` の連続した並び (n-gram) のうち、2回以上現れるものの割合**を返す。

```
repetition_rate('a b c a b c a b c', n=3)
  3-gram: (a,b,c) (b,c,a) (c,a,b) (a,b,c) (b,c,a) (c,a,b) (a,b,c)   全7個
  2回以上現れる位置: 全部 → 1.0
```

- n-gram が1つも作れないときは `0.0` を返す
- 返り値は `0.0` 以上 `1.0` 以下の `float`

**ヒント**: `Counter` で n-gram を数え、`count >= 2` のものが全体のいくつを占めるか。

### 例題

次のセルを実行して、動きを見る。

n-gram の作り方と数え方。


In [ ]:
# 例題
from collections import Counter

ws = 'a b c a b'.split()
grams = [tuple(ws[i:i + 2]) for i in range(len(ws) - 2 + 1)]
print(grams)          # [('a','b'), ('b','c'), ('c','a'), ('a','b')]
print(Counter(grams)) # Counter({('a','b'): 2, ('b','c'): 1, ('c','a'): 1})


あとは「`count >= 2` の gram が、全 4 個のうちいくつを占めるか」。
この例なら 2/4 = 0.5。
`len(ws) - n + 1` が 0 以下のときは gram が作れないので `0.0` を返す。


In [ ]:
from collections import Counter


def repetition_rate(text, n=3):
    # TODO
    pass


# 実際に測ってみる
out = generate_greedy('好きな食べ物は', 80)
print(out)
print('繰り返し率:', repetition_rate(out))


In [ ]:
# [CHECK] q2b
need('repetition_rate', arity=2)

assert repetition_rate('a b c a b c a b c', n=3) == 1.0
assert repetition_rate('a b c d e f g', n=3) == 0.0
assert repetition_rate('', n=3) == 0.0
assert repetition_rate('a b', n=3) == 0.0, 'n-gram が作れないときは 0.0'
assert repetition_rate('a b c', n=3) == 0.0, '1個だけなら繰り返しではない'

r = repetition_rate('x y z x y z p q r', n=3)
assert 0.0 < r < 1.0, r
assert abs(r - 2/7) < 1e-9, f'{r} (7個の3-gram のうち (x,y,z) だけが2回)'

assert repetition_rate('a a a a', n=1) == 1.0
assert isinstance(repetition_rate('a b c a b c', n=2), float)

# 性質: 割合なので必ず 0 以上 1 以下
holds('0 以上 1 以下',
      lambda ws: 0.0 <= repetition_rate(' '.join(ws), 2) <= 1.0,
      [words(n) for n in range(0, 12)])
holds('同じ語だけ並べたら 1.0',
      lambda n: repetition_rate(' '.join(['x'] * n), 2) == 1.0, range(4, 12))
print('OK')


---
## 課題3: 温度付き top-k サンプリング (20点)

`sample_next(logits, temperature, k)` を書け。

1. `logits / temperature` を softmax して確率にする
2. 上位 `k` 個だけを候補にする
3. その確率に比例して1つ引く

**ヒント**: `torch.multinomial(weights, 1)` が重み付きのくじ引き。
返るのは「上位 k 個の中での位置」なので、元の ID に戻す必要がある。

### 例題

次のセルを実行して、動きを見る。

くじ引きの部分。**引いた結果は「上位 k 個の中での位置」**なので、
元のトークン ID に戻す一手間が要る。ここが一番間違えやすい。


In [ ]:
# 例題
lg = next_logits(encode('日本の首都は'))

probs = torch.softmax(lg / 0.8, dim=-1)      # 温度で割ってから softmax
v, idx = torch.topk(probs, 5)                # v=確率, idx=元のID
print(idx)                                   # tensor([100562, 3583, ...])

pos = torch.multinomial(v, 1)                # 0〜4 の「位置」が返る
print(pos)                                   # tensor([2])
print(idx[pos])                              # tensor([...])  ← これが答え


温度の効き方はこうなる。


In [ ]:
# 例題
lg2 = torch.tensor([3.0, 1.0])
print(torch.softmax(lg2 / 0.1, dim=-1))      # [1.00, 0.00] 低温は決め打ち
print(torch.softmax(lg2 / 2.0, dim=-1))      # [0.73, 0.27] 高温はばらける


In [ ]:
def sample_next(logits, temperature=0.8, k=50):
    # TODO
    pass


def generate(prompt, max_new_tokens=60, temperature=0.8, k=50):
    ids = encode(prompt)
    for _ in range(max_new_tokens):
        next_id = sample_next(next_logits(ids), temperature, k)
        if next_id.item() == tok.eos_token_id:
            break
        ids = torch.cat([ids, next_id.view(1, 1)], dim=1)
    return tok.decode(ids[0], skip_special_tokens=True)


for t in [0.3, 0.8, 1.5]:
    print(f'--- temperature = {t} ---')
    print(generate('プログラミングを学ぶと', max_new_tokens=40, temperature=t))
    print()


In [ ]:
# [CHECK] q3
need('sample_next', arity=3)

logits = next_logits(encode('日本の首都は'))

# k=1 なら greedy と一致するはず
for _ in range(5):
    assert sample_next(logits, temperature=1.0, k=1).item() == torch.argmax(logits).item()

# 候補は必ず上位 k 個の中から出る
topk_ids = set(torch.topk(logits, 50).indices.tolist())
for _ in range(30):
    assert sample_next(logits, 1.0, 50).item() in topk_ids
top5 = set(torch.topk(logits, 5).indices.tolist())
for _ in range(30):
    assert sample_next(logits, 1.0, 5).item() in top5, 'k が効いていない'

# 温度を上げれば、選ばれるトークンの種類が増えるはず
cold = {sample_next(logits, 0.1, 50).item() for _ in range(30)}
hot  = {sample_next(logits, 2.0, 50).item() for _ in range(30)}
assert len(hot) > len(cold), f'低温 {len(cold)} 種 / 高温 {len(hot)} 種'

# 生成が動くこと
g = generate('むかしむかし', max_new_tokens=10, temperature=0.8)
assert isinstance(g, str) and g.startswith('むかしむかし')
print('OK')


---
## 課題4: chat (10点)

チャットテンプレートを使って、1回の応答を返す `chat(messages)` を書け。

1. `tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)` で
   プロンプト文字列を作る
2. `sample_next` を使って生成する
3. **新しく生成した部分だけ**を文字列で返す

**ヒント**: 生成前の `ids.shape[1]` を覚えておき、最後に `ids[0, start:]` を decode する。
停止判定は `next_id.item() in tok.all_special_ids`。

### 例題

次のセルを実行して、動きを見る。

まず、テンプレートが何を作るのかを見る。


In [ ]:
# 例題
p = tok.apply_chat_template(
    [{'role': 'user', 'content': 'やあ'}],
    tokenize=False, add_generation_prompt=True)
print(repr(p))
# '<|im_start|>system\n...<|im_end|>\n<|im_start|>user\nやあ<|im_end|>\n
#  <|im_start|>assistant\n'


つまり `chat` は「この文字列の続きを生成する」だけ。
生成のループは課題3 の `generate` とまったく同じ。違いは2つ。


In [ ]:
# 例題
start = ids.shape[1]                    # 生成前の長さを覚えておく
# ... ここで課題3 と同じループ ...
tok.decode(ids[0, start:], skip_special_tokens=True)   # 増えたぶんだけ返す


止まる判定は `if next_id.item() in tok.all_special_ids: break`。
`<|im_end|>` も特殊トークンなので、これで応答の終わりを捕まえられる。


In [ ]:
def chat(messages, max_new_tokens=60, temperature=0.7, k=50):
    prompt = tok.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)
    ids = encode(prompt)
    start = ids.shape[1]
    # TODO
    return tok.decode(ids[0, start:], skip_special_tokens=True)


msgs = [
    {'role': 'system', 'content': 'あなたは簡潔に答えるアシスタントです。'},
    {'role': 'user', 'content': '日本の首都はどこですか？'},
]
print(chat(msgs))


In [ ]:
# [CHECK] q4
need('chat', arity=1)

reply = chat([{'role': 'user', 'content': '1 + 1 は？'}], max_new_tokens=30)
assert isinstance(reply, str) and reply.strip(), '空の応答'
assert '1 + 1 は？' not in reply, 'プロンプトが混ざっている (start の位置を見直す)'
assert '<|im_start|>' not in reply and '<|im_end|>' not in reply, '特殊トークンが残っている'
assert 'system' not in reply.lower()[:20], 'テンプレート全体を返している'

# max_new_tokens を守ること
short = chat([{'role': 'user', 'content': '長い昔話をして'}], max_new_tokens=5)
assert len(tok(short).input_ids) <= 6, f'{len(tok(short).input_ids)} トークン返った'
print('OK')


### 課題4-b: 会話を続ける (10点)

`chat_session(user_lines)` を書け。
ユーザの発言のリストを順に投げ、**履歴を積み重ねながら**応答を集めて返す。

- 返り値は応答文字列のリスト (長さは `user_lines` と同じ)
- 2回目以降は、それまでの `user` と `assistant` の発言をすべて含めて `chat` に渡す
- system メッセージは先頭に1つだけ

**採点できるように `input()` は使わないこと。** 対話したいときは下のセルを使う。

**ヒント**: 履歴のリストに `{'role': 'user', ...}` と
`{'role': 'assistant', ...}` を交互に足していく。

### 例題

次のセルを実行して、動きを見る。

履歴に交互に足していくだけ。


In [ ]:
# 例題
history = [{'role': 'system', 'content': SYSTEM}]

history.append({'role': 'user', 'content': 'こんにちは'})
r = chat(history)                                   # 履歴ぜんぶを渡す
history.append({'role': 'assistant', 'content': r})

history.append({'role': 'user', 'content': '元気？'})
r2 = chat(history)                                  # 今度は4件ぶん渡る


課題は、これを `for u in user_lines:` のループにしたもの。
`replies` には `r` だけを足していく。


In [ ]:
SYSTEM = '簡潔に答えてください。'


def chat_session(user_lines, max_new_tokens=40):
    history = [{'role': 'system', 'content': SYSTEM}]
    replies = []
    # TODO
    return replies


for u, r in zip(['犬と猫はどちらが好き？', 'なぜ？', '他には？'],
                chat_session(['犬と猫はどちらが好き？', 'なぜ？', '他には？'])):
    print('あなた:', u)
    print('AI    :', r)
    print()


In [ ]:
# 対話用 (採点しない)。実行すると入力待ち
def chat_interactive():
    history = [{'role': 'system', 'content': SYSTEM}]
    while True:
        user = input('あなた: ')
        if user.strip() in ('quit', 'exit', ''):
            break
        history.append({'role': 'user', 'content': user})
        reply = chat(history)
        print('AI:', reply)
        history.append({'role': 'assistant', 'content': reply})


# chat_interactive()   ← コメントを外すと対話できる


In [ ]:
# [CHECK] q4b
need('chat_session', arity=1)

rs = chat_session(['こんにちは', '元気？', 'さようなら'], max_new_tokens=20)
assert isinstance(rs, list) and len(rs) == 3, f'{len(rs)} 件返った'
assert all(isinstance(r, str) and r.strip() for r in rs), '空の応答がある'
assert all('<|im_' not in r for r in rs), '特殊トークンが残っている'
assert chat_session([]) == []

# 履歴が積み上がっていることを、渡された messages の長さで確かめる
seen = []
_orig = chat
def _spy(messages, **kw):
    seen.append([m['role'] for m in messages])
    return _orig(messages, **kw)
chat = _spy
try:
    chat_session(['A', 'B', 'C'], max_new_tokens=8)
finally:
    chat = _orig
assert len(seen) == 3, f'chat が {len(seen)} 回しか呼ばれていない'
assert seen[0].count('user') == 1, seen[0]
assert seen[1].count('user') == 2 and seen[1].count('assistant') == 1, seen[1]
assert seen[2].count('user') == 3 and seen[2].count('assistant') == 2, seen[2]
assert all(r[0] == 'system' and r.count('system') == 1 for r in seen), 'system は先頭に1つ'
print('OK')


---
# 発展課題

余力があれば。


## 発展1: top-p (nucleus) サンプリング (5点)

確率の高い順に足していき、累積が `p` を超える手前までを候補にする
`sample_next_top_p(logits, temperature, p)` を書け。

**ヒント**: `torch.sort(probs, descending=True)` → `torch.cumsum` →
`cumsum - sorted_probs < p` でマスクを作る (最上位が必ず残るように)。

### 例題

次のセルを実行して、動きを見る。

累積を見て、切る位置を決める。


In [ ]:
# 例題
probs = torch.tensor([0.5, 0.3, 0.15, 0.05])
cum = torch.cumsum(probs, dim=-1)
print(cum)                        # [0.50, 0.80, 0.95, 1.00]
print(cum - probs)                # [0.00, 0.50, 0.80, 0.95]  自分より前の合計
print((cum - probs) < 0.9)        # [True, True, True, False]


`cum` ではなく `cum - probs` (自分より前までの合計) と比べるのがコツ。
こうすると、最上位は必ず `True` になるので候補が空にならない。
あとは残った確率で `multinomial` を引き、課題3 と同じように元の ID に戻す。


In [ ]:
def sample_next_top_p(logits, temperature=0.8, p=0.9):
    # TODO
    pass


In [ ]:
# [CHECK] b1
need('sample_next_top_p', arity=3)

logits = next_logits(encode('日本の首都は'))
probs = torch.softmax(logits, dim=-1)

# p が極小なら、実質 greedy
for _ in range(5):
    assert sample_next_top_p(logits, 1.0, 1e-6).item() == torch.argmax(logits).item()

# p=0.9 の候補は、累積 0.9 に届くまでの語だけ
sp, si = torch.sort(probs, descending=True)
n_keep = int((torch.cumsum(sp, -1) - sp < 0.9).sum())
allowed = set(si[:n_keep].tolist())
for _ in range(30):
    assert sample_next_top_p(logits, 1.0, 0.9).item() in allowed
print(f'OK (候補は {n_keep} 語)')


## 発展2: 繰り返しペナルティ (5点)

すでに出たトークンを選びにくくする `apply_penalty(logits, used_ids, penalty)` を書け。
**新しい logits を返す** (元は書き換えない)。

`used_ids` に含まれる位置について:

- logits が正なら `penalty` で**割る**
- logits が負なら `penalty` を**掛ける**

どちらも「値を下げる」向き。負の値を割ると 0 に近づいてしまうので掛ける。

**ヒント**: `out = logits.clone()` してから、該当位置だけ書き換える。

### 例題

次のセルを実行して、動きを見る。

まとめて書き換える。


In [ ]:
# 例題
lg = torch.tensor([2.0, -2.0, 0.0, 5.0])
out = lg.clone()                        # 元を壊さない
ids = torch.tensor([0, 1])

sel = out[ids]                          # 対象だけ取り出す
out[ids] = torch.where(sel > 0, sel / 2.0, sel * 2.0)
print(out)                              # [1., -4., 0., 5.]
print(lg)                               # [2., -2., 0., 5.]  元は無事


`for` で1つずつ書き換えてもよい。
`torch.where(条件, 真のとき, 偽のとき)` は要素ごとの三項演算子。


In [ ]:
def apply_penalty(logits, used_ids, penalty=1.2):
    # TODO
    pass


def generate_penalized(prompt, max_new_tokens=80, penalty=1.2, temperature=0.8):
    ids = encode(prompt)
    for _ in range(max_new_tokens):
        lg = apply_penalty(next_logits(ids), ids[0].tolist(), penalty)
        next_id = sample_next(lg, temperature, 50)
        if next_id.item() == tok.eos_token_id:
            break
        ids = torch.cat([ids, next_id.view(1, 1)], dim=1)
    return tok.decode(ids[0], skip_special_tokens=True)


out = generate_penalized('好きな食べ物は')
print(out)
print('繰り返し率:', repetition_rate(out))


In [ ]:
# [CHECK] b2
need('apply_penalty', arity=3)

lg = torch.tensor([2.0, -2.0, 0.0, 5.0])
out = apply_penalty(lg, [0, 1], 2.0)
assert out is not lg, '元の logits を書き換えないこと'
assert abs(float(out[0]) - 1.0) < 1e-6, f'正の値は割る: {float(out[0])}'
assert abs(float(out[1]) - (-4.0)) < 1e-6, f'負の値は掛ける: {float(out[1])}'
assert abs(float(out[2]) - 0.0) < 1e-6
assert abs(float(out[3]) - 5.0) < 1e-6, '対象外の位置は変えないこと'
assert abs(float(lg[0]) - 2.0) < 1e-6, '元のテンソルが書き換わっている'

# penalty=1.0 なら何も変わらない
same = apply_penalty(lg, [0, 1, 2, 3], 1.0)
assert torch.allclose(same, lg)

# 使われたトークンの順位は下がる
top = int(torch.argmax(lg))
assert int(torch.argmax(apply_penalty(lg, [top], 10.0))) != top
print('OK')


---
## 自己確認

`score()` で全部まとめて確かめる。何度やってもよい。
`NG` の行に、次に何をすればいいかが出る。


In [ ]:
# [SKIP] 採点の対象外
score()


---
## 提出

提出は kibaco から。`score()` と採点ページは下書き確認用。

1. 上から実行し、出力を残す
2. `ファイル → .ipynb をダウンロード`
3. 締切までに kibaco へ出す

確認セルが全部 `OK` なら提出してよい。確認セルを消しても教員の採点は別の検査で行う。
